# Salecta: run from a browser (no local setup)

Use this when you don't have your own computer set up. It needs **no GPU**. Click **Runtime → Run all**
(Colab) or **Run All** (Kaggle, with *Internet: On*), paste your OpenRouter key when asked, and download
the zip at the end.

Pick what to run in the second cell: the full Salecta run (3 models, 180 prompts each), or grading
of refusal-direction results.

In [ ]:
!git clone -q https://github.com/Lord-Shax/Salecta.git 2>/dev/null || (cd Salecta && git pull -q)
%cd Salecta
!pip -q install -U -e ".[dev,providers]" openai inspect-ai 2>&1 | grep -v "dependency\|requires\|incompatible" || true
import os
from getpass import getpass
# Always ask, so a wrong key from an earlier attempt is replaced. An OpenRouter key starts with "sk-or-".
while True:
    key = getpass("Paste your OpenRouter key (starts with sk-or-), then press Enter: ").strip().strip('"').strip("'")
    if key.startswith("sk-or-"):
        break
    print("That is not an OpenRouter key" + (" (it looks like a Hugging Face token)" if key.startswith("hf_") else "")
          + ". Copy it from openrouter.ai/settings/keys and try again.")
os.environ["OPENROUTER_API_KEY"] = key
print("OpenRouter key accepted.")

## A. Full Salecta run (3 models)
Takes roughly 10–20 minutes. Safe to re-run: finished models are skipped.

In [ ]:
!python scripts/full_run.py

In [ ]:
# Download the results (Colab). On Kaggle, find salecta_full_run.zip in the Output panel instead.
try:
    from google.colab import files
    files.download("salecta_full_run.zip")
except ImportError:
    import shutil; shutil.copy("salecta_full_run.zip", "/kaggle/working/") if os.path.isdir("/kaggle/working") else None
    print("Saved salecta_full_run.zip")

## B. Grade refusal-direction results
Skip section A for this (or let it run; it is harmless). Run this box: a **Choose files** button appears.
Pick all your `results_<model>.zip` files at once. It grades every reply and downloads
`graded_results.zip` at the end.

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile
zips = sorted(set(glob.glob("results_*.zip") + glob.glob("/kaggle/input/**/results_*.zip", recursive=True)))
if not zips:
    try:
        from google.colab import files
        print("Choose your results_<model>.zip files (you can pick several at once):")
        zips = sorted(files.upload().keys())
    except ImportError:
        print("Add the zips to this folder (Kaggle: Add Input) and run again.")
print("Grading:", zips)
for z in zips:
    local = os.path.basename(z)
    if os.path.abspath(z) != os.path.abspath(local):
        shutil.copy(z, local)
    subprocess.run([sys.executable, "scripts/grade_replies.py", local], check=False)
graded = sorted(glob.glob("graded_*.csv"))
with zipfile.ZipFile("graded_results.zip", "w", zipfile.ZIP_DEFLATED) as out:
    for g in graded:
        out.write(g)
print("Graded files:", graded)
try:
    from google.colab import files
    files.download("graded_results.zip")
except ImportError:
    print("Saved graded_results.zip")